In [6]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin
from IPython.display import Image, display  

# 1. 뉴스 URL
news_url = 'https://news.nate.com/recent?mid=n0300'  # '경제' 섹션
res = requests.get(news_url)
res.encoding = 'utf-8'
print(f"응답 코드: {res.status_code}")

# 2. HTML 파싱
if res.ok:
    soup = BeautifulSoup(res.text, 'html.parser')

    # 뉴스 블록 선택
    news_list = soup.select("div.postList > ul > li")

    # 이미지 저장 폴더 생성
    img_folder = 'img'
    if not os.path.isdir(img_folder):
        os.mkdir(img_folder)

    # 3. 뉴스 반복하며 데이터 추출
    for news in news_list[:5]:  # 5개만 예시로
        # 제목과 링크
        a_tag = news.select_one("a.thumb")
        if not a_tag:
            continue

        title_tag = news.select_one("strong")
        title = title_tag.text.strip() if title_tag else "제목 없음"
        link = urljoin("https://news.nate.com", a_tag["href"])

        print(f"\n 제목: {title}")
        print(f" 링크: {link}")

        # 이미지 처리
        img_tag = news.select_one("img")
        if img_tag and 'src' in img_tag.attrs:
            src = img_tag["src"].strip()
            if src.startswith("//"):
                src = "https:" + src
            else:
                src = urljoin("https://news.nate.com", src)

            # 이미지 다운로드
            img_res = requests.get(src)
            if img_res.ok:
                img_data = img_res.content
                file_name = os.path.join(img_folder, os.path.basename(src))
                with open(file_name, 'wb') as f:
                    f.write(img_data)
                    print(f" 이미지 저장: {file_name} ({len(img_data):,} bytes)")

                #  이미지 화면에 출력
                display(Image(data=img_data))
            else:
                print(f" 이미지 요청 실패: {img_res.status_code}")
        else:
            print(" [이미지 없음]")

응답 코드: 200


In [7]:
print()